# Go2 Track 2 Bonus Project
**Team: Jiarao_Zhang** | EEC289A/EEC289Q

HW1 `best_checkpoint` is already committed to the repo — **no low-level retraining needed**.
Run cells top-to-bottom. MLP artifacts are auto-saved to Google Drive.

| Step | Cell | Time |
|------|------|------|
| Mount Drive + Config | 1 | 1 min |
| Install + Clone repos | 2 | 5–10 min |
| Copy Go2 assets | 3 | 1 min |
| Patch planner.py (VX_MAX→0.85) | 4 | instant |
| Configure runtime + dry-run | 5 | instant |
| Verify HW1 checkpoint | 6 | instant |
| Quick smoke test | 7 | 3–5 min |
| CMA-ES MLP training (in-process) | 8 | 30–60 min |
| Save MLP to Drive | 9 | instant |
| Full track eval + video | 10 | 5–10 min |
| Create submission.json | 11 | instant |
| Copy artifacts + final checklist | 12 | instant |
| Push to GitHub | 13 | 1 min |

In [ ]:
# ── CELL 1: Mount Drive + Config ─────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os, sys, shutil, subprocess, json, io, tarfile, tempfile, time, urllib.request
from pathlib import Path
from urllib.parse import urlparse

TEAM_NAME            = "Jiarao_Zhang"
COURSE_REPO_URL      = "https://github.com/jiarao76/Final-Project-Track-2-Bonus-Project.git"
COURSE_REPO_BRANCH   = "main"

PLAYGROUND_REPO      = "https://github.com/google-deepmind/mujoco_playground.git"
PLAYGROUND_REF       = "dd38c285c6d54266287081e516109f0b15985818"
UNITREE_MUJOCO_REPO  = "https://github.com/unitreerobotics/unitree_mujoco.git"
UNITREE_MUJOCO_REF   = "1a37b051a10be723405b7ed6dc839361af036d88"
MENAGERIE_REPO       = "https://github.com/deepmind/mujoco_menagerie.git"
MENAGERIE_REF        = "1b86ece576591213e2b666ebf59508454200ca97"

BASE_DIR             = Path("/content")
COURSE_REPO_DIR      = BASE_DIR / "go2_track_bonus_repo"
PLAYGROUND_DIR       = BASE_DIR / "mujoco_playground"
UNITREE_DIR          = BASE_DIR / "unitree_mujoco"
MENAGERIE_DIR        = PLAYGROUND_DIR / "mujoco_playground" / "external_deps" / "mujoco_menagerie"

# HW1 checkpoint lives in the course repo root (already committed)
CHECKPOINT_DIR       = COURSE_REPO_DIR / "best_checkpoint"

DRIVE_BACKUP         = Path("/content/drive/MyDrive/go2_track_backup")
DRIVE_BACKUP.mkdir(parents=True, exist_ok=True)

print("Drive backup dir :", DRIVE_BACKUP)
print("Course repo dir  :", COURSE_REPO_DIR)
print("Checkpoint dir   :", CHECKPOINT_DIR)

def run(cmd):
    cmd = [str(c) for c in cmd]
    print("+", " ".join(cmd))
    subprocess.run(cmd, check=True)

def github_archive_url(repo_url, ref):
    repo_path = urlparse(repo_url).path.strip("/")
    if repo_path.endswith(".git"):
        repo_path = repo_path[:-4]
    return f"https://codeload.github.com/{repo_path}/tar.gz/{ref}"

def download_repo_snapshot(repo_url, ref, target_dir):
    archive_url = github_archive_url(repo_url, ref)
    print(f"+ download {archive_url}")
    target_dir.parent.mkdir(parents=True, exist_ok=True)
    tmp_dir = Path(tempfile.mkdtemp(prefix=f"{target_dir.name}_", dir=str(target_dir.parent)))
    try:
        with urllib.request.urlopen(archive_url) as response:
            payload = response.read()
        with tarfile.open(fileobj=io.BytesIO(payload), mode="r:gz") as archive:
            archive.extractall(tmp_dir)
        extracted_dirs = [p for p in tmp_dir.iterdir() if p.is_dir()]
        if len(extracted_dirs) != 1:
            raise RuntimeError(f"Expected one extracted directory, got {extracted_dirs}")
        if target_dir.exists():
            shutil.rmtree(target_dir)
        shutil.move(str(extracted_dirs[0]), str(target_dir))
    finally:
        shutil.rmtree(tmp_dir, ignore_errors=True)

def ensure_pinned_repo(repo_url, ref, target_dir):
    if target_dir.exists() and (target_dir / ".git").exists():
        try:
            run(["git", "-C", target_dir, "fetch", "--all", "--tags"])
            run(["git", "-C", target_dir, "checkout", ref])
            return
        except subprocess.CalledProcessError:
            shutil.rmtree(target_dir)
    elif target_dir.exists():
        shutil.rmtree(target_dir)
    try:
        run(["git", "clone", repo_url, target_dir])
        run(["git", "-C", target_dir, "checkout", ref])
    except subprocess.CalledProcessError:
        if target_dir.exists():
            shutil.rmtree(target_dir)
        download_repo_snapshot(repo_url, ref, target_dir)

def ensure_course_repo(repo_url, branch, target_dir, reset=False):
    if target_dir.exists():
        if reset:
            shutil.rmtree(target_dir)
        else:
            print(f"+ reuse existing course repo at {target_dir}")
            return
    try:
        run(["git", "clone", repo_url, target_dir])
    except subprocess.CalledProcessError:
        if target_dir.exists():
            shutil.rmtree(target_dir)
        download_repo_snapshot(repo_url, branch, target_dir)

print("Cell 1 done.")

In [ ]:
# ── CELL 2: Install packages + Clone repos ────────────────────────────────────
if shutil.which("ffmpeg") is None:
    run(["apt-get", "update", "-qq"])
    run(["apt-get", "install", "-y", "ffmpeg"])

ensure_pinned_repo(PLAYGROUND_REPO,     PLAYGROUND_REF,     PLAYGROUND_DIR)
ensure_pinned_repo(UNITREE_MUJOCO_REPO, UNITREE_MUJOCO_REF, UNITREE_DIR)
ensure_pinned_repo(MENAGERIE_REPO,      MENAGERIE_REF,      MENAGERIE_DIR)
ensure_course_repo(COURSE_REPO_URL, COURSE_REPO_BRANCH, COURSE_REPO_DIR)

os.chdir(COURSE_REPO_DIR)
!python -m pip install -q -U pip setuptools wheel
!python -m pip uninstall -y playground 2>/dev/null || true
!python -m pip install -q -r {COURSE_REPO_DIR / 'configs' / 'colab_requirements.txt'}

os.chdir(PLAYGROUND_DIR)
!python -m pip install -q -e .
os.chdir(COURSE_REPO_DIR)

if str(PLAYGROUND_DIR.resolve()) not in sys.path:
    sys.path.insert(0, str(PLAYGROUND_DIR.resolve()))
if str(COURSE_REPO_DIR.resolve()) not in sys.path:
    sys.path.insert(0, str(COURSE_REPO_DIR.resolve()))

import jax
import mujoco_playground
print("JAX devices:", jax.devices())
print("JAX backend:", jax.default_backend())

expected_playground = str(PLAYGROUND_DIR.resolve())
if expected_playground not in str(Path(mujoco_playground.__file__).resolve()):
    raise RuntimeError("mujoco_playground imported from wrong location")
print("Cell 2 done.")

In [ ]:
# ── CELL 3: Copy Go2 assets ───────────────────────────────────────────────────
os.chdir(COURSE_REPO_DIR)
!python scripts/copy_go2_assets.py \
    --unitree-dir {UNITREE_DIR} \
    --course-dir {COURSE_REPO_DIR}
print("Assets copied.")

In [ ]:
# ── CELL 4: Patch planner.py – raise speed limits ────────────────────────────
# The repo has VX_MAX=0.50 m/s; the HW1 policy handles up to ~0.95 m/s.
# We raise VX_MAX to 0.85 so the MLP can command faster forward speeds.
planner_py = COURSE_REPO_DIR / "track_bonus" / "planner.py"
content = planner_py.read_text()

replacements = {
    "_VX_MAX: float = 0.50": "_VX_MAX: float = 0.85",
    "_VY_LIM: float = 0.10": "_VY_LIM: float = 0.20",
    "_YAW_LIM: float = 0.30": "_YAW_LIM: float = 0.45",
}
for old, new in replacements.items():
    if old in content:
        content = content.replace(old, new)
        print(f"Patched: {old} → {new}")
    else:
        print(f"[ok] already patched or not found: {old}")
planner_py.write_text(content)

for line in content.splitlines():
    if any(k in line for k in ["_VX_MAX", "_VX_MIN", "_VY_LIM", "_YAW_LIM"]):
        print(" ", line.strip())

In [ ]:
# ── CELL 5: Configure runtime + dry-run ──────────────────────────────────────
import json
os.chdir(COURSE_REPO_DIR)

runtime_config = {
    "num_envs": 1024,
    "num_eval_envs": 128,
    "num_evals": 5,
    "batch_size": 256,
    "policy_hidden_layer_sizes": [256, 256, 128],
    "value_hidden_layer_sizes": [256, 256, 128],
    "stage_1_num_timesteps": 10_000_000,
    "stage_2_num_timesteps": 5_000_000,
}

config_path   = COURSE_REPO_DIR / "configs" / "colab_runtime_config.json"
base_cfg_path = COURSE_REPO_DIR / "configs" / "course_config.json"
base_config   = json.loads(base_cfg_path.read_text())
base_config["runtime_overrides"] = runtime_config
config_path.write_text(json.dumps(base_config, indent=2))
print("Config written:", config_path)

!python train.py --config {config_path} --dry-run

In [ ]:
# ── CELL 6: Verify HW1 checkpoint ────────────────────────────────────────────
# The checkpoint was committed to the repo root as best_checkpoint/
# No retraining needed.
import json
CHECKPOINT_DIR = COURSE_REPO_DIR / "best_checkpoint"

if not CHECKPOINT_DIR.exists():
    raise FileNotFoundError(
        f"Checkpoint missing at {CHECKPOINT_DIR}. "
        "Make sure 'git clone' completed successfully in Cell 2."
    )

cfg = json.loads((CHECKPOINT_DIR / "ppo_network_config.json").read_text())
kwargs = cfg.get("network_factory_kwargs", {})
print("Checkpoint OK")
print("  policy_obs_key :", kwargs.get("policy_obs_key"))
print("  action_size    :", cfg.get("action_size"))
print("  files          :", [f.name for f in CHECKPOINT_DIR.iterdir()])

In [ ]:
# ── CELL 7: Quick smoke test (10 s, no render) ────────────────────────────────
PLANNER_CONFIG = COURSE_REPO_DIR / "configs" / "starter_planner.json"
SMOKE_DIR      = COURSE_REPO_DIR / "artifacts" / "smoke_test"

os.chdir(COURSE_REPO_DIR)
!python run_track_bonus.py \
    --checkpoint-dir {CHECKPOINT_DIR} \
    --planner-config {PLANNER_CONFIG} \
    --config configs/colab_runtime_config.json \
    --output-dir {SMOKE_DIR} \
    --entry-name {TEAM_NAME} \
    --duration-seconds 10 \
    --no-render

if (SMOKE_DIR / "results.json").exists():
    r = json.loads((SMOKE_DIR / "results.json").read_text())
    print("Smoke metrics:", {k: round(v, 3) if isinstance(v, float) else v
                              for k, v in r["metrics"].items()})
    print("Policy is working!")
else:
    print("[error] results.json not found – check errors above")

In [ ]:
# ── CELL 8: In-process CMA-ES MLP training (~30-60 min) ──────────────────────
#
# Key design: env + policy loaded ONCE, all candidates evaluated in-process.
# First call pays JIT compilation (~2-3 min), subsequent calls ~10-30 s each.
# With 10 gens × 8 pop = 80 candidates, total ≈ 30-60 min.

import numpy as np, time, json

if str(COURSE_REPO_DIR.resolve()) not in sys.path:
    sys.path.insert(0, str(COURSE_REPO_DIR.resolve()))

from track_bonus.planner import MLPTrackPlanner, _VX_MAX, _VX_MIN, _VY_LIM, _YAW_LIM
from track_bonus.official_track import official_track
from track_bonus.scoring import compute_track_bonus_metrics
from course_common import lazy_import_stack, load_json, set_runtime_env
from run_track_bonus import rollout, _make_env
from test_policy import load_policy_with_workaround

CHECKPOINT_DIR = COURSE_REPO_DIR / "best_checkpoint"
HIGHLEVEL_DIR  = COURSE_REPO_DIR / "artifacts" / "highlevel_mlp"
HIGHLEVEL_DIR.mkdir(parents=True, exist_ok=True)

print(f"Speed limits: VX=[{_VX_MIN}, {_VX_MAX}]  VY=±{_VY_LIM}  YAW=±{_YAW_LIM}")

# ── Hyper-params ──────────────────────────────────────────────────────────────
HIDDEN_SIZES  = [32, 16]
N_GENERATIONS = 10
POPULATION    = 8
SIGMA0        = 0.30
SEED          = 42
EVAL_SECONDS  = 350.0  # sim seconds (enough for 1 full lap at ≥0.6 m/s avg)

# ── Load env + policy ONCE ────────────────────────────────────────────────────
print("\nLoading env + policy...")
set_runtime_env()
course_cfg = load_json(COURSE_REPO_DIR / "configs" / "colab_runtime_config.json")
course_cfg["runtime_overrides"] = {}
num_steps = int(round(EVAL_SECONDS / course_cfg["control"]["ctrl_dt"]))
track  = official_track()
stack  = lazy_import_stack()
env    = _make_env(stack, course_cfg, "stage_2", num_steps)
policy = load_policy_with_workaround(CHECKPOINT_DIR.resolve(), deterministic=True)
policy = stack["jax"].jit(policy)
print(f"Ready. num_steps={num_steps}")

# ── Fitness ───────────────────────────────────────────────────────────────────
def evaluate(theta, seed=SEED):
    weights = MLPTrackPlanner.unpack(theta, HIDDEN_SIZES)
    planner = MLPTrackPlanner(weights, HIDDEN_SIZES, stand_seconds=1.0)
    result  = rollout(
        stack=stack, env=env, policy=policy, planner=planner,
        track=track, num_steps=num_steps, seed=seed, start_s=0.0, force_cpu=False,
    )
    m    = compute_track_bonus_metrics(result, track)
    dist = m["valid_distance_m"] / 200.0
    fall = m.get("fall", True)
    ft   = m.get("finish_time")
    fit  = min(dist, 1.0)
    if ft is not None:
        fit = 1.0 + max(0.0, (350.0 - float(ft)) / 350.0)
    if fall:
        fit *= 0.55
    return float(fit), m

# ── CMA-ES ────────────────────────────────────────────────────────────────────
class _CMAes:
    def __init__(self, x0, sigma0=0.3, popsize=8, seed=0):
        self.rng = np.random.default_rng(seed)
        self.n   = len(x0)
        self.mean  = x0.copy().astype(np.float64)
        self.sigma = float(sigma0)
        self.lam   = popsize
        self.mu    = max(popsize // 2, 2)
        raw_w = np.log(self.mu + 0.5) - np.log(np.arange(1, self.mu + 1))
        self.w = raw_w / raw_w.sum()
        self.mueff = 1.0 / float(np.sum(self.w ** 2))
        self.cs = (self.mueff + 2.0) / (self.n + self.mueff + 5.0)
        self.ds = 1.0 + 2.0 * max(0.0, np.sqrt((self.mueff-1.0)/(self.n+1.0))-1.0) + self.cs
        self.chiN = float(np.sqrt(self.n) * (1.0 - 1.0/(4.0*self.n) + 1.0/(21.0*self.n**2)))
        self.ps  = np.zeros(self.n)
        self.var = np.ones(self.n)

    def ask(self):
        return self.mean + self.sigma * np.sqrt(self.var) * self.rng.standard_normal((self.lam, self.n))

    def tell(self, xs, scores):
        order    = np.argsort(-scores)
        elite    = xs[order[:self.mu]]
        old_mean = self.mean.copy()
        self.mean = (self.w[:, None] * elite).sum(axis=0)
        step = (self.mean - old_mean) / (self.sigma * np.sqrt(self.var) + 1e-12)
        self.ps = (1-self.cs)*self.ps + np.sqrt(self.cs*(2-self.cs)*self.mueff)*step
        self.sigma *= float(np.exp((self.cs/self.ds)*(np.linalg.norm(self.ps)/self.chiN-1.0)))
        self.sigma  = float(np.clip(self.sigma, 1e-8, 2.0))
        ys = (elite - old_mean) / (self.sigma * np.sqrt(self.var) + 1e-12)
        self.var = np.clip(0.9*self.var + 0.1*float(np.sum(self.w))*(self.w[:,None]*ys**2).sum(0), 1e-10, None)

# ── Init ──────────────────────────────────────────────────────────────────────
n_params = MLPTrackPlanner.param_count(HIDDEN_SIZES)
print(f"\nMLP 5→{HIDDEN_SIZES}→3  ({n_params} params)")
print(f"CMA-ES: {N_GENERATIONS} gens × {POPULATION} pop  eval={EVAL_SECONDS}s")

theta0     = MLPTrackPlanner.pack(MLPTrackPlanner.make_weights(HIDDEN_SIZES, seed=SEED))
es         = _CMAes(theta0, sigma0=SIGMA0, popsize=POPULATION, seed=SEED)
best_score = -1.0
best_theta = theta0.copy()
history    = []

print("\nWarm-up rollout (triggers JAX JIT, ~2-3 min)...")
t0 = time.time()
_, wm = evaluate(theta0, seed=SEED)
print(f"Warm-up done in {time.time()-t0:.1f}s")
print(f"  distance={wm['valid_distance_m']:.1f}m  fall={wm.get('fall')}  finish_time={wm.get('finish_time')}")

# ── Loop ──────────────────────────────────────────────────────────────────────
for gen in range(N_GENERATIONS):
    t_gen      = time.time()
    candidates = es.ask()
    scores     = np.zeros(POPULATION)
    print(f"\n── Gen {gen+1}/{N_GENERATIONS}  σ={es.sigma:.4f} ──")

    for idx, theta in enumerate(candidates):
        t0 = time.time()
        score, m = evaluate(theta, seed=SEED + gen*100 + idx)
        scores[idx] = score
        ft  = m.get("finish_time")
        tag = f"LAP {ft:.1f}s" if ft is not None else f"dist={m['valid_distance_m']:.1f}m"
        star = "★" if score > best_score else " "
        print(f"  {star} [{idx+1}/{POPULATION}] fit={score:.4f}  {tag}  fall={m.get('fall')}  ({time.time()-t0:.1f}s)")
        if score > best_score:
            best_score = score
            best_theta = theta.copy()
            np.savez(str(HIGHLEVEL_DIR / "planner_weights.npz"),
                     **MLPTrackPlanner.unpack(best_theta, HIDDEN_SIZES))

    es.tell(candidates, scores)
    gen_t = time.time() - t_gen
    history.append({"gen": gen, "best": float(best_score),
                    "max": float(scores.max()), "mean": float(scores.mean()),
                    "sigma": float(es.sigma), "time_s": gen_t})
    print(f"  Best={best_score:.4f}  GenMax={scores.max():.4f}  ({gen_t:.0f}s)")
    (HIGHLEVEL_DIR / "search_history.json").write_text(json.dumps(history, indent=2))

# ── Save config ───────────────────────────────────────────────────────────────
final_cfg = {
    "planner_type":     "mlp",
    "mlp_weights_path": "planner_weights.npz",
    "mlp_hidden":       HIDDEN_SIZES,
    "stand_seconds":    1.0,
    "training_info":    {
        "vx_max": _VX_MAX, "vx_min": _VX_MIN,
        "vy_lim": _VY_LIM, "yaw_lim": _YAW_LIM,
        "best_fitness": float(best_score),
        "generations": N_GENERATIONS, "population": POPULATION,
    },
}
(HIGHLEVEL_DIR / "planner_config.json").write_text(json.dumps(final_cfg, indent=2))
print(f"\nDone. Best fitness={best_score:.4f}")
print("fitness > 1.0 means at least one lap completed with speed bonus")

In [ ]:
# ── CELL 9: Save MLP artifacts to Drive ──────────────────────────────────────
HIGHLEVEL_DIR = COURSE_REPO_DIR / "artifacts" / "highlevel_mlp"
drive_mlp     = DRIVE_BACKUP / "highlevel_mlp"

if drive_mlp.exists():
    shutil.rmtree(drive_mlp)
shutil.copytree(str(HIGHLEVEL_DIR), str(drive_mlp))
print("Saved to Drive:", drive_mlp)
for f in drive_mlp.iterdir():
    print(" ", f.name)

# ── Restore from Drive (if session restarted before Cell 10) ─────────────────
# Uncomment and run if HIGHLEVEL_DIR is missing after a restart:
# if not HIGHLEVEL_DIR.exists() and drive_mlp.exists():
#     shutil.copytree(str(drive_mlp), str(HIGHLEVEL_DIR))
#     print("Restored from Drive.")

In [ ]:
# ── CELL 10: Full track evaluation (300 s, with video) ───────────────────────
CHECKPOINT_DIR = COURSE_REPO_DIR / "best_checkpoint"
PLANNER_CONFIG = COURSE_REPO_DIR / "artifacts" / "highlevel_mlp" / "planner_config.json"
TRACK_EVAL_DIR = COURSE_REPO_DIR / "artifacts" / "track_eval"

os.chdir(COURSE_REPO_DIR)
!python run_track_bonus.py \
    --checkpoint-dir {CHECKPOINT_DIR} \
    --planner-config {PLANNER_CONFIG} \
    --config configs/colab_runtime_config.json \
    --output-dir {TRACK_EVAL_DIR} \
    --entry-name {TEAM_NAME} \
    --duration-seconds 300 \
    --render-every 10 \
    --render-fps 5

if (TRACK_EVAL_DIR / "results.json").exists():
    r = json.loads((TRACK_EVAL_DIR / "results.json").read_text())
    m = r["metrics"]
    s = r["scores"]
    print("\n=== RESULTS ===")
    print(f"  composite_score  : {s['composite_score']:.4f}")
    print(f"  lap_completion   : {m['lap_completion']}")
    print(f"  finish_time      : {m['finish_time']}")
    print(f"  mean_speed       : {m['mean_progress_speed']:.3f} m/s")
    print(f"  fall             : {m['fall']}")
    print(f"  rms_lateral_err  : {m['rms_lateral_error']:.3f} m")
else:
    print("[error] results.json not found")

In [ ]:
# ── CELL 11: Create submission.json ──────────────────────────────────────────
import json
submission = {
    "team_name": TEAM_NAME,
    "track2_option": "leaderboard",
    "checkpoint_dir": "best_checkpoint",
    "planner_config": "planner_config.json",
    "planner_code": "track_bonus/planner.py",
    "planner_weights": "planner_weights.npz",
    "high_level_planner_type": "learned_mlp_cmaes",
    "track_eval": "track_eval/results.json",
    "notes": (
        "Low-level: reused HW1 best_checkpoint (PPO stage1+stage2, action_size=12). "
        "High-level: 5->32->16->3 MLP (VX_MAX=0.85 m/s, VY=0.20, YAW=0.45) trained with "
        "diagonal CMA-ES (10 gen x 8 pop), in-process evaluation (JIT compiled once), "
        "fitness = dist/200 + speed_bonus if lap completed. "
        "Failed attempts: subprocess-based CMA-ES (120s JIT per candidate), "
        "training faster low-level policy from scratch (unstable at 1.2+ m/s), "
        "fine-tuning via --restore-checkpoint-dir (wrong checkpoint format)."
    ),
}
(COURSE_REPO_DIR / "submission.json").write_text(json.dumps(submission, indent=2))
print((COURSE_REPO_DIR / "submission.json").read_text())

In [ ]:
# ── CELL 12: Copy artifacts to repo root + final checklist ───────────────────
HIGHLEVEL_DIR  = COURSE_REPO_DIR / "artifacts" / "highlevel_mlp"
TRACK_EVAL_DIR = COURSE_REPO_DIR / "artifacts" / "track_eval"

# planner_config.json + planner_weights.npz → repo root
shutil.copy(str(HIGHLEVEL_DIR / "planner_config.json"), str(COURSE_REPO_DIR / "planner_config.json"))
shutil.copy(str(HIGHLEVEL_DIR / "planner_weights.npz"), str(COURSE_REPO_DIR / "planner_weights.npz"))

# track_eval/ → repo root
dest_eval = COURSE_REPO_DIR / "track_eval"
if dest_eval.exists():
    shutil.rmtree(dest_eval)
shutil.copytree(str(TRACK_EVAL_DIR), str(dest_eval))

# Checklist
expected = {
    "best_checkpoint/"        : COURSE_REPO_DIR / "best_checkpoint",
    "planner_config.json"     : COURSE_REPO_DIR / "planner_config.json",
    "planner_weights.npz"     : COURSE_REPO_DIR / "planner_weights.npz",
    "track_bonus/planner.py"  : COURSE_REPO_DIR / "track_bonus" / "planner.py",
    "track_eval/results.json" : COURSE_REPO_DIR / "track_eval" / "results.json",
    "submission.json"         : COURSE_REPO_DIR / "submission.json",
}
all_ok = True
for label, path in expected.items():
    ok = path.exists()
    if not ok:
        all_ok = False
    print("OK  " if ok else "MISS", label)

if all_ok:
    # Package to Drive
    drive_final = DRIVE_BACKUP / "final_submission"
    if drive_final.exists():
        shutil.rmtree(drive_final)
    drive_final.mkdir(parents=True)
    for label, path in expected.items():
        dest = drive_final / label.rstrip("/")
        dest.parent.mkdir(parents=True, exist_ok=True)
        if path.is_dir():
            shutil.copytree(str(path), str(dest))
        else:
            shutil.copy(str(path), str(dest))
    print(f"\nPackaged to Drive: {drive_final}")

    results_path = COURSE_REPO_DIR / "track_eval" / "results.json"
    if results_path.exists():
        r = json.loads(results_path.read_text())
        print(f"\ncomposite_score : {r['scores']['composite_score']:.4f}")
        print(f"finish_time     : {r['metrics']['finish_time']}")
        print(f"fall            : {r['metrics']['fall']}")

In [ ]:
# ── CELL 13: Push results to GitHub ──────────────────────────────────────────
# Run after all artifacts are ready.
import getpass
os.chdir(COURSE_REPO_DIR)

!git config --global user.email "jrzzhang@ucdavis.edu"
!git config --global user.name "Jiarao Zhang"

# Stage everything
!git add -f submission.json track_bonus/planner.py planner_config.json planner_weights.npz
!git add -f best_checkpoint/ track_eval/ 2>/dev/null || true
!git status --short

token = getpass.getpass("GitHub token (repo scope): ")
REMOTE = f"https://{token}@github.com/jiarao76/Final-Project-Track-2-Bonus-Project.git"

!git commit -m "Add MLP planner weights, track eval results, and submission metadata"
!git push {REMOTE} main
print("Pushed to GitHub.")